In [5]:
!git clone https://github.com/gkianfar/TIHM-Dataset-Visualization.git

Cloning into 'TIHM-Dataset-Visualization'...
remote: Enumerating objects: 337, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 337 (delta 7), reused 0 (delta 0), pack-reused 322 (from 1)
Receiving objects: 100% (337/337), 42.68 MiB | 20.13 MiB/s, done.
Resolving deltas: 100% (189/189), done.
Updating files: 100% (16/16), done.


In [5]:
!rm -r /content/TIHM

Load TIHM Dataset

In [6]:
%cd /content/TIHM-Dataset-Visualization
from utils import load_datasets
# Load dataset
activity_df, physiology_df, sleep_df, labels_df, demographics_df =\
 load_datasets('/content/TIHM-Dataset-Visualization/Data')

/content/TIHM-Dataset-Visualization


Load Modal

*As it is difficult, to load the original Llama model, we load the Tiny version*




In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from huggingface_hub import login

device = 'cpu'

# Load TinyLlama model and tokenizer
model_name = "meta-llama/Llama-3.2-1B"
#model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map=device)

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Fix 1: Set pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def ask_question(question, instruction=None, context=None):
    # Formulate the prompt
    prompt = ""

    if instruction:
        prompt += f"### Instruction:\n{instruction}\n\n"

    if context:
        prompt += f"### Context:\n{context}\n\n"

    prompt += f"### Question:\n{question}\n\n### Answer:\n"

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate an answer
    with torch.no_grad():
        # Fix 2: Add generation parameters
        output_ids = model.generate(
            **inputs,
            max_new_tokens=500,          # Increased from default
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id  # Fix 3: Explicit pad token
        )

    # Fix 4: Proper output decoding
    answer = tokenizer.decode(
        output_ids[0][inputs.input_ids.shape[-1]:],  # Skip input tokens
        skip_special_tokens=True
    ).strip()

    return answer

# Fix 5: Corrected function name (typo fix)
question = """You are a virtual tour guide from 1901. You have tourists visiting Eiffel Tower. Describe Eiffel Tower to your audience. Begin with
1. Why it was built
2. Then by how long it took them to build
3. Where were the materials sourced to build
4. Number of people it took to build
5. End it with the number of people visiting the Eiffel tour annually in the 1900's, the amount of time it completes a full tour and why so many people visit this place each year.
Make your tour funny by including 1 or 2 funny jokes at the end of the tour.

"""

answer = ask_question(question)  # Fixed function name
print("Answer:", answer)

Answer: I will answer these questions one-by-one:

Why It Was Built: The first reason is because it has been constructed for such an important event that made history known as "EIFFEL TOWER". Second Reason is based on its construction process which includes steel beams, metal girders etc. Thirdly,it was very easy to construct due to large quantity of raw material available there.

Then By How Long Took Them To Build : I think they could take up least 10 days just to complete their task but we cannot really say since no record exists regarding any work done on Eiffel tower till date

Where Were Materials Sourced? We can assume it might be France where most likely everything required for building would come under control here like iron ore mines,silicon ore deposits,iron works,hollow sections & tubes,bars,wire ropes,cables,& other related products used during construction period.

Number Of People Who Completed Building In One Day Is Unknowable But As Per My Understanding They Might Have

RAG Enhanced

In [21]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 18.6 MB/s eta 0:00:00


In [22]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample knowledge base
documents = [
    "Deep Q-Learning is a reinforcement learning technique.",
    "UAVs communicate over sub-6 GHz channels.",
    "Transformers use self-attention for sequence modeling."
]

# Convert text to embeddings
doc_embeddings = model.encode(documents)

# Create FAISS index
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings).astype('float32'))

def retrieve_relevant_docs(query, top_k=2):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), top_k)
    return [documents[i] for i in indices[0]]

# Example Query
query = "How do UAVs communicate?"
retrieved_docs = retrieve_relevant_docs(query)
print("Retrieved:", retrieved_docs)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieved: ['UAVs communicate over sub-6 GHz channels.', 'Transformers use self-attention for sequence modeling.']


In [24]:
index.data

TypeError: 'IndexFlatL2' object is not callable

In [4]:
from transformers import pipeline

# Initialize the pipeline with LLaMA 3.1 70B Instruct model
pipe = pipeline("text-generation", model=model_name)

messages = [
    {"role": "user", "content": "Who are you?"},
]

# Generate the response
response = pipe(messages)
print(response)

Device set to use cpu


ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templating

{'<unk>': 0,
 '<s>': 1,
 '</s>': 2,
 '<0x00>': 3,
 '<0x01>': 4,
 '<0x02>': 5,
 '<0x03>': 6,
 '<0x04>': 7,
 '<0x05>': 8,
 '<0x06>': 9,
 '<0x07>': 10,
 '<0x08>': 11,
 '<0x09>': 12,
 '<0x0A>': 13,
 '<0x0B>': 14,
 '<0x0C>': 15,
 '<0x0D>': 16,
 '<0x0E>': 17,
 '<0x0F>': 18,
 '<0x10>': 19,
 '<0x11>': 20,
 '<0x12>': 21,
 '<0x13>': 22,
 '<0x14>': 23,
 '<0x15>': 24,
 '<0x16>': 25,
 '<0x17>': 26,
 '<0x18>': 27,
 '<0x19>': 28,
 '<0x1A>': 29,
 '<0x1B>': 30,
 '<0x1C>': 31,
 '<0x1D>': 32,
 '<0x1E>': 33,
 '<0x1F>': 34,
 '<0x20>': 35,
 '<0x21>': 36,
 '<0x22>': 37,
 '<0x23>': 38,
 '<0x24>': 39,
 '<0x25>': 40,
 '<0x26>': 41,
 '<0x27>': 42,
 '<0x28>': 43,
 '<0x29>': 44,
 '<0x2A>': 45,
 '<0x2B>': 46,
 '<0x2C>': 47,
 '<0x2D>': 48,
 '<0x2E>': 49,
 '<0x2F>': 50,
 '<0x30>': 51,
 '<0x31>': 52,
 '<0x32>': 53,
 '<0x33>': 54,
 '<0x34>': 55,
 '<0x35>': 56,
 '<0x36>': 57,
 '<0x37>': 58,
 '<0x38>': 59,
 '<0x39>': 60,
 '<0x3A>': 61,
 '<0x3B>': 62,
 '<0x3C>': 63,
 '<0x3D>': 64,
 '<0x3E>': 65,
 '<0x3F>': 66,
 '<0x40>': 